# Ensemble Learning - Bagging Companion Notebook

This notebook is the practical code companion for the bagging part of [`lecture_notes/12_ensemble_learning.pdf`](../../lecture_notes/12_ensemble_learning.pdf). The lecture notes cover the concepts, derivations, and comparisons; the goal here is to make the main mechanisms visible in runnable experiments.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import BaggingClassifier, ExtraTreesClassifier, RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42


This setup cell imports the models and evaluation utilities used throughout the notebook. The experiments use breast cancer classification because it is small, fast to run, and expressive enough to show how bagging stabilizes a high-variance learner.


## Bootstrap Sampling in Practice

Before fitting an ensemble, it is useful to simulate what a bootstrap sample actually looks like.


In [ ]:
rng = np.random.default_rng(SEED)
m = 500
n_bootstraps = 1000

unique_fractions = []
for _ in range(n_bootstraps):
    sample_idx = rng.integers(0, m, size=m)
    unique_fractions.append(len(np.unique(sample_idx)) / m)

bootstrap_summary = pd.DataFrame({
    "empirical_mean_unique_fraction": [np.mean(unique_fractions)],
    "theoretical_limit": [1 - np.exp(-1)],
}).round(4)
bootstrap_summary


This simulation estimates the fraction of unique examples present in a bootstrap sample of size `m`. The empirical average should land close to the theoretical value `1 - e^(-1) ≈ 0.632`, which is the key quantity discussed in the notes.


In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(unique_fractions, bins=30, edgecolor="black", color="tab:blue", alpha=0.8)
plt.axvline(1 - np.exp(-1), color="tab:red", linestyle="--", label="1 - exp(-1)")
plt.xlabel("fraction of unique examples in one bootstrap sample")
plt.ylabel("count")
plt.title("Bootstrap samples concentrate near the 63.2% rule")
plt.legend()
plt.show()


The histogram is useful because it turns the 63.2% rule into something concrete. Each draw is slightly different, but the distribution concentrates around the same value, which explains why bagging repeatedly sees perturbed versions of the training set rather than completely new datasets.


## Compare a Single Tree, Bagging, and Random Forest

The central bagging question is whether aggregating unstable learners produces a more reliable predictor than a single tree. This comparison also includes `ExtraTrees`, a closely related randomized-tree ensemble that increases diversity through additional randomization in the split selection step.


In [ ]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

tree = DecisionTreeClassifier(random_state=SEED)
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=SEED),
    n_estimators=100,
    bootstrap=True,
    oob_score=True,
    random_state=SEED,
)
forest = RandomForestClassifier(
    n_estimators=100,
    oob_score=True,
    random_state=SEED,
)
extra_trees = ExtraTreesClassifier(
    n_estimators=100,
    random_state=SEED,
)

models = {
    "Decision Tree": tree,
    "Bagging": bagging,
    "Random Forest": forest,
    "Extra Trees": extra_trees,
}


This cell defines four related models. The single decision tree is the unstable baseline, bagging reduces variance by bootstrap aggregation, random forest adds feature subsampling, and extra trees adds even more randomization to decorrelate the trees further.


In [ ]:
cv_rows = []
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="accuracy")
    cv_rows.append({
        "model": name,
        "cv_mean_accuracy": scores.mean(),
        "cv_std": scores.std(),
    })

pd.DataFrame(cv_rows).sort_values("cv_mean_accuracy", ascending=False).round(4)


Cross-validation gives a fair first comparison because all models are evaluated on the same training split under the same procedure. The average accuracy summarizes performance, while the standard deviation gives a sense of stability across folds.


In [ ]:
test_rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    row = {
        "model": name,
        "test_accuracy": accuracy_score(y_test, y_pred),
    }
    if hasattr(model, "oob_score_"):
        row["oob_score"] = model.oob_score_
    test_rows.append(row)

pd.DataFrame(test_rows).round(4)


This table moves from cross-validation to one final held-out test comparison. For bagging and random forest, the OOB score is shown alongside test accuracy because it acts as a built-in validation estimate and is one of the most practical features of bagged ensembles.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

for ax, (name, model) in zip(axes, models.items()):
    ConfusionMatrixDisplay.from_estimator(model, X_test, y_test, ax=ax, colorbar=False)
    ax.set_title(name)

plt.tight_layout()
plt.show()


The confusion matrices are useful because two models with similar overall accuracy can still make different types of mistakes. Reading these side by side helps you see whether the ensemble is just slightly improving counts or actually changing the error pattern in a meaningful way.


## OOB Error as the Ensemble Grows

A final practical question is how the out-of-bag estimate evolves as more trees are added.


In [ ]:
ensemble_sizes = [5, 10, 20, 40, 80, 120, 160]
oob_rows = []

for n_estimators in ensemble_sizes:
    model = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED),
        n_estimators=n_estimators,
        bootstrap=True,
        oob_score=True,
        random_state=SEED,
    )
    model.fit(X_train, y_train)
    oob_rows.append({
        "n_estimators": n_estimators,
        "oob_error": 1 - model.oob_score_,
    })

oob_df = pd.DataFrame(oob_rows)
oob_df


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(oob_df["n_estimators"], oob_df["oob_error"], marker="o")
plt.xlabel("number of estimators")
plt.ylabel("OOB error")
plt.title("Bagging OOB error typically stabilizes as the ensemble grows")
plt.show()


This last plot is a practical diagnostic. It shows whether adding more estimators is still materially changing the OOB estimate. Once the curve stabilizes, the ensemble has usually reached the regime where extra trees mostly increase cost rather than predictive benefit.
